In [ ]:
!pip install --upgrade torchao

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 47.4 MB/s eta 0:00:00
  Attempting uninstall: torchao
    Found existing installation: torchao 0.10.0
    Uninstalling torchao-0.10.0:
      Successfully uninstalled torchao-0.10.0


In [ ]:
import json, math, torch
import torch.nn as nn
from pathlib import Path
from huggingface_hub import snapshot_download
from transformers import AutoTokenizer, Qwen3Model
from peft import PeftModel

REPO_ID = "Piyush-5541/bev-decision-lora"
HF_TOKEN = None
DEVICE = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"
TASK_TYPES = {"choice": 0, "noul": 1, "score": 2}
PROMPTS = {"choice": "Choose the best option.", "score": "Rate it on the given scale.", "noul": "Answer yes or no."}


class SA(nn.Module):
    def __init__(self, h, d):
        super().__init__(); self.d = d; self.q = nn.Linear(h, d); self.k = nn.Linear(h, d); self.v = nn.Linear(h, d); self.o = nn.Linear(d, h)
    def forward(self, x, valid):
        w = torch.softmax((torch.einsum("bqd,bkd->bqk", self.q(x), self.k(x)) / math.sqrt(self.d)).masked_fill(~valid[:, None, :], -float("inf")), dim=-1)
        return self.o(torch.einsum("bqk,bkd->bqd", w, self.v(x)))

class Block(nn.Module):
    def __init__(self, h, d):
        super().__init__(); self.n1 = nn.LayerNorm(h); self.a = SA(h, d); self.n2 = nn.LayerNorm(h); self.m = nn.Sequential(nn.Linear(h, d), nn.GELU(), nn.Linear(d, h))
    def forward(self, x, v):
        x = x + self.a(self.n1(x), v); return x + self.m(self.n2(x))

class Head(nn.Module):
    def __init__(self, h, new_dim, num_layers, num_task_types):
        super().__init__(); self.d = new_dim; self.task_embedding = nn.Embedding(num_task_types, h); self.norm = nn.LayerNorm(h)
        self.blocks = nn.ModuleList([Block(h, new_dim) for _ in range(num_layers)]); self.final = nn.LayerNorm(h); self.ap = nn.Linear(h, new_dim); self.cp = nn.Linear(h, new_dim)
    def forward(self, choices, answer, choice_mask, task):
        x = self.norm(torch.cat([self.task_embedding(task).unsqueeze(1), choices, answer.unsqueeze(1)], 1))
        valid = torch.ones((len(x), x.shape[1]), dtype=torch.bool, device=x.device)
        for b in self.blocks: x = b(x, valid)
        x = self.final(x)
        return torch.einsum("bd,bcd->bc", self.ap(x[:, -1]), self.cp(x[:, 1:-1])) / math.sqrt(self.d)

# 2. Fetch Model via Hugging Face Hub API
folder = Path(snapshot_download(repo_id=REPO_ID, token=HF_TOKEN))
meta = json.loads((folder / "meta.json").read_text())
tok = AutoTokenizer.from_pretrained(meta["model_name"])

base = Qwen3Model.from_pretrained(meta["model_name"], dtype=torch.float32, num_hidden_layers=meta["num_layers"])
if (folder / "lora").exists():
    base = PeftModel.from_pretrained(base, folder / "lora")
head = Head(base.config.hidden_size, **meta["head_config"])
head.load_state_dict(torch.load(folder / "head.pt", map_location="cpu"))
base.to(DEVICE).eval(); head.to(DEVICE).eval()

# 3. Single-Sample Inference Function
@torch.no_grad()
def ask_bev(state, question, options, kind="choice"):
    q_txt = f"{PROMPTS[kind]}\nQuestion: {question}\n\n{state}\n"
    tokens = tok.encode(q_txt); q_end = len(tokens) - 1; ends = []
    b_tok, c_tok = tok.encode("<option>"), tok.encode("</option>")

    for opt in options:
        tokens += b_tok + tok.encode(opt) + c_tok; ends.append(len(tokens) - 1)
    tokens += tok.encode("The answer is:"); ans_end = len(tokens) - 1

    # Parallel positions & 2D mask so options don't see each other
    pos = list(range(q_end + 1)); base_p = next_s = len(pos)
    for e in ends: pos += list(range(base_p, base_p + e - next_s + 1)); next_s = e + 1
    pos += list(range(max(pos) + 1, max(pos) + 1 + ans_end + 1 - len(pos)))

    N = len(tokens); mask = torch.zeros((N, N)); q_len = q_end + 1
    mask[:q_len, :q_len] = torch.tril(torch.ones(q_len, q_len)); s = q_len
    for e in ends: w = e - s + 1; mask[s:e+1, s:e+1] = torch.tril(torch.ones(w, w)); s = e + 1
    mask[q_len:, :q_len] = 1; mask[s:, :s] = 1; mask[s:, s:] = torch.tril(torch.ones(N - s, N - s))

    # Forward pass through Qwen + LoRA + Custom Head
    h = base(
        input_ids=torch.tensor([tokens], device=DEVICE),
        position_ids=torch.tensor([pos], device=DEVICE),
        attention_mask=torch.where(mask == 0, -float("inf"), 0.0).view(1, 1, N, N).to(DEVICE),
        use_cache=False
    ).last_hidden_state.float()

    read_idx = [x - len(c_tok) for x in ends]
    logits = head(
        choices=h[:, read_idx],
        answer=h[:, ans_end],
        choice_mask=torch.ones((1, len(options)), dtype=torch.bool, device=DEVICE),
        task=torch.tensor([TASK_TYPES[kind]], device=DEVICE)
    )[0]

    probs = torch.softmax(logits, dim=-1).tolist()
    best = int(logits.argmax().item())
    return best, probs



Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 8 files:   0%|          | 0/8 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/222 [00:00<?, ?it/s]

[transformers] Qwen3Model LOAD REPORT from: Qwen/Qwen3-0.6B
Key                                                                           | Status     |  | 
------------------------------------------------------------------------------+------------+--+-
model.layers.{20, 21, 22, 23, 24, 25, 26, 27}.mlp.up_proj.weight              | UNEXPECTED |  | 
model.layers.{20, 21, 22, 23, 24, 25, 26, 27}.self_attn.q_norm.weight         | UNEXPECTED |  | 
model.layers.{20, 21, 22, 23, 24, 25, 26, 27}.self_attn.k_norm.weight         | UNEXPECTED |  | 
model.layers.{20, 21, 22, 23, 24, 25, 26, 27}.self_attn.q_proj.weight         | UNEXPECTED |  | 
model.layers.{20, 21, 22, 23, 24, 25, 26, 27}.self_attn.k_proj.weight         | UNEXPECTED |  | 
model.layers.{20, 21, 22, 23, 24, 25, 26, 27}.post_attention_layernorm.weight | UNEXPECTED |  | 
model.layers.{20, 21, 22, 23, 24, 25, 26, 27}.mlp.gate_proj.weight            | UNEXPECTED |  | 
model.layers.{20, 21, 22, 23, 24, 25, 26, 27}.input_layernorm.weigh


Question: Which blocks contain medium yellow squares?
          [0] ( 10.8%): B
          [1] (  0.4%): C
          [2] ( 35.7%): A and B
          [3] ( 10.8%): A and C
          [4] (  1.3%): None
          [5] (  0.4%): B and C
          [6] (  0.7%): A and B and C
✅ WINNER [7] ( 39.9%): A


In [ ]:
# =========================================================
# 4. PUT YOUR QUESTION AND OPTIONS HERE
# =========================================================
state = (

)
question = "What is the capital of India?"
options = ["Mumbai", "Kolkata", "New Delhi", "Chennai"]

best_idx, probs = ask_bev(state, question, options, kind="choice")

print(f"\nQuestion: {question}")
for i, (opt, p) in enumerate(zip(options, probs)):
    tag = "✅ WINNER" if i == best_idx else ""
    print(f"{tag} [{i}] ({p*100:5.1f}%): {opt}")


Question: What is the capital of India?
 [0] ( 19.4%): Mumbai
 [1] ( 17.0%): Kolkata
✅ WINNER [2] ( 56.2%): New Delhi
 [3] (  7.5%): Chennai
